# 01 · Build the tables (ERA5 features + labels)
For every month: load ERA5 (main + buffer strips) → compute features on the 0.25° grid → take 0.5° cells →
attach labels → keep sea / near-sea cells → subsample easy negatives (weights keep probabilities correct).
The same pass also writes the **storm-centred ERA5 features** used by notebook 02.

**Runtime:** CPU + **High-RAM**. **Takes:** several hours for 1990–2023 (4 months in parallel).
Months already built by an earlier run (e.g. 1990–2008) are **skipped** – their tables do not depend on the split.
The extra recent test months (`C.RECENT_TEST_MONTHS`, Oct–Dec 2025) are built by their own cell in Step 3.
**Safe to re-run:** finished months are skipped, so after a disconnect just run all cells again.

**Outputs**
* `outputs/tables/occurrence/YYYY_MM` – training rows (column `w` = sample weight)
* `outputs/tables/occurrence_full/YYYY_MM` – every cell at 00/12 UTC, validation, calibration and test years
* `outputs/tables/storm_env/YYYY_MM` – storm-centred features
* `outputs/tables/storm_rows` – storm skeleton with history features and targets

In [ ]:
# ▶ Setup: connect Google Drive, install missing packages, load the project code
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost", "lightgbm", "xgboost"], check=False)

PROJECT_DIR = "/content/drive/MyDrive/CycloneProject"      # <- change only if you put the project elsewhere
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import numpy as np, pandas as pd
from src import config as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)
print("project:", C.PROJECT, "| ERA5:", C.ERA5_ROOT)

In [ ]:
# ▶ Use a LOCAL copy of the code, so worker processes never import from Drive
import shutil
shutil.copytree(f"{PROJECT_DIR}/src", "/content/code/src", dirs_exist_ok=True)
for k in [k for k in sys.modules if k == "src" or k.startswith("src.")]:
    del sys.modules[k]
sys.path.insert(0, "/content/code")
from src import config as C
print(f"{shutil.disk_usage('/content').free / 1e9:.0f} GB free on local disk (need ~30 GB)")

In [ ]:
# ▶ Load IBTrACS, build hourly tracks and the storm skeleton
from src import tracks, labels, build_month as B
from src.common import save_table, load_table, months

fixes = tracks.load_ibtracs()
TRK, INFO = tracks.build_tracks(fixes)
TP = tracks.TrackPoints(TRK)
ROWS = labels.storm_rows(TRK, INFO)
save_table(ROWS, C.out("tables") / "storm_rows")
B.init(TP, ROWS)
print(f"{len(TRK)} storms, {len(ROWS)} storm rows (3-hourly, depression or stronger)")
print(ROWS.split.value_counts())

## Step 1 · Test on 2 months first
May 2020 (cyclone Amphan) and May 2021 (cyclone Tauktae). Check the printout:
* positives should show **higher** `vort850_max100` and **lower** `msla_min100` than negatives
* NaN columns: `sst_c` / `airsea_dt` are NaN over land (fine); `up24_*` is NaN near the domain edge (fine)

In [ ]:
TEST_MONTHS = [(2020, 5), (2021, 5)]          # Amphan, Tauktae
log = B.run_all(TEST_MONTHS, n_workers=2, force=True)
display(log[["year", "month", "status", "seconds"]])
for y, m in TEST_MONTHS:
    print(f"\n==== {y}-{m:02d} ====")
    B.sanity_report(load_table(B.paths(y, m)["occ"]))
sec = log.loc[log.status == "ok", "seconds"].mean()
from src.common import all_months
n = len(all_months())
print(f"\n~{sec:.0f} s per month -> about {sec * n / C.N_WORKERS / 3600:.1f} h for {n} months with {C.N_WORKERS} workers")

## Step 2 · Full run (one year at a time)
If Drive disconnects (`Errno 107`): **Runtime → Disconnect and delete runtime**, then run the setup cell,
the local-code cell, the IBTrACS cell and this cell again. It continues where it stopped.

In [ ]:
# ▶ Full run, one YEAR at a time:
#   1) copy that year's ERA5 files Drive -> local disk, one file at a time (gentle on Drive)
#   2) 4 workers build the tables from the local files -> saved to Drive (outputs/tables)
#   3) delete the local copies (December is kept for next January)
# Safe to re-run: finished months are skipped.
import time, os
from pathlib import Path
from src import io_era5 as IO
IO.clear_local = lambda y, m: None          # this cell deletes local files itself, per year

def copy_with_retry(src, dst, tries=5):
    for k in range(tries):
        try:
            dst.parent.mkdir(parents=True, exist_ok=True)
            tmp = dst.with_name(dst.name + ".part")
            shutil.copyfile(src, tmp); tmp.replace(dst)
            return
        except OSError as e:
            print(f"  retry {k+1}: {src.name} ({e})"); time.sleep(30)
    raise RuntimeError(f"Drive keeps failing on {src} - Runtime > Disconnect and delete runtime, then re-run")

def prefetch(y, m):
    for p in IO.month_files(y, m).values():
        if p is not None:
            dst = Path(C.LOCAL_CACHE) / Path(p).relative_to(C.ERA5_ROOT)
            if not dst.exists():
                copy_with_retry(Path(p), dst)

def drop_cache(y, m):
    for p in Path(C.LOCAL_CACHE).rglob(f"*{y}_{m:02d}*.nc"):
        p.unlink(missing_ok=True)

for y in range(C.YEARS[0], C.YEARS[1] + 1):
    todo = [(y, m) for m in range(1, 13) if not B.is_done(y, m)]
    if not todo:
        print(y, "already done"); continue
    os.listdir(C.ERA5_ROOT)                  # fails fast if Drive died
    t = time.time()
    if y > C.YEARS[0]:
        prefetch(y - 1, 12)                  # December tail for January
    for m in range(1, 13):
        prefetch(y, m)
    print(f"{y}: copied in {(time.time() - t) / 60:.1f} min")
    B.run_all(todo, n_workers=C.N_WORKERS, log_path=C.out("checks") / "build_log.csv")
    if y > C.YEARS[0]:
        drop_cache(y - 1, 12)
    for m in range(1, 12):
        drop_cache(y, m)
    print(f"{y}: finished in {(time.time() - t) / 60:.1f} min\n")

## Step 3 · Recent test months (Montha, Ditwah)
Same processing, only the months in `C.RECENT_TEST_MONTHS`. If the month before is not on Drive, the first day's pressure-fall / rain features are NaN (a warning, not an error).

In [ ]:
todo = [(y, m) for (y, m) in C.RECENT_TEST_MONTHS if not B.is_done(y, m)]
if not todo:
    print("recent test months already done")
else:
    os.listdir(C.ERA5_ROOT)
    t = time.time()
    for y, m in todo:
        py, pm = (y, m - 1) if m > 1 else (y - 1, 12)
        prefetch(py, pm)                     # last 24 h of the previous month (skipped if not downloaded)
        prefetch(y, m)
    B.run_all(todo, n_workers=min(C.N_WORKERS, len(todo)), log_path=C.out("checks") / "build_log.csv")
    for y, m in todo:
        drop_cache(y, m)
    print(f"recent test months finished in {(time.time() - t) / 60:.1f} min")

In [ ]:
# ▶ Summary of what is on disk
from src.common import list_tables
for sub in ("occurrence", "occurrence_full", "storm_env"):
    files = list_tables(C.out("tables", sub))
    print(f"{sub:17s}: {len(files)} monthly files")
from src.common import all_months
todo = [(y, m) for (y, m) in all_months() if not B.is_done(y, m)]
print("months still missing:", todo[:20], "..." if len(todo) > 20 else "")

**Next:** `02_build_storm_table.ipynb` (fast), then `03` and `04`.